<a href="https://colab.research.google.com/github/jacobwechuli/QA-intelligent-RAG-system/blob/main/Full_RAG_Chatbot_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pymupdf pdf2image pytesseract sentence-transformers faiss-cpu gradio huggingface_hub
!apt-get -qq install -y poppler-utils tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.7 MB/s eta 0:00:00
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/p/poppler/poppler-utils_22.02.0-2ubuntu0.12_amd64.deb  404  Not Found [IP: 91.189.91.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [ ]:
import fitz  # PyMuPDF
import pytesseract
from pdf2image import convert_from_path
import os

MIN_TEXT_LEN = 20  # below this, treat page as "scanned" and OCR it

def process_pdf(pdf_path, source_id=None, doc_type="Unknown"):
    """
    Returns a list of dicts: {text, page, source, doc_type}
    Falls back to OCR per-page when digital text extraction is empty/too short.
    """
    source_id = source_id or os.path.basename(pdf_path)
    doc = fitz.open(pdf_path)
    pages_out = []

    ocr_page_indices = []
    for i in range(doc.page_count):
        text = doc.load_page(i).get_text("text").strip()
        if len(text) < MIN_TEXT_LEN:
            ocr_page_indices.append(i)
            pages_out.append({"text": "", "page": i, "source": source_id, "doc_type": doc_type})
        else:
            pages_out.append({"text": text, "page": i, "source": source_id, "doc_type": doc_type})

    if ocr_page_indices:
        images = convert_from_path(pdf_path, dpi=200)
        for i in ocr_page_indices:
            ocr_text = pytesseract.image_to_string(images[i])
            pages_out[i]["text"] = ocr_text.strip()

    doc.close()
    return pages_out

In [ ]:
def chunk_pages(pages, chunk_size=800, overlap=100):
    """
    Splits page text into overlapping chunks, tagging each with
    doc_type, page range, and source identifier.
    """
    chunks = []
    for p in pages:
        text = p["text"]
        if not text:
            continue
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]
            chunks.append({
                "text": chunk_text,
                "doc_type": p["doc_type"],
                "source": p["source"],
                "page_start": p["page"],
                "page_end": p["page"],
            })
            start += chunk_size - overlap
    return chunks

def build_corpus(pdf_paths_with_meta):
    """
    pdf_paths_with_meta: list of (path, source_id, doc_type)
    """
    all_chunks = []
    for path, source_id, doc_type in pdf_paths_with_meta:
        pages = process_pdf(path, source_id=source_id, doc_type=doc_type)
        all_chunks.extend(chunk_pages(pages))
    return all_chunks

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # free, open-source
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

class VectorStore:
    def __init__(self):
        self.index = None
        self.chunks = []  # parallel metadata list

    def build(self, chunks):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        embeddings = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
        embeddings = np.array(embeddings, dtype="float32")
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors = cosine sim
        self.index.add(embeddings)

    def search(self, query, k=4):
        q_emb = embed_model.encode([query], normalize_embeddings=True)
        q_emb = np.array(q_emb, dtype="float32")
        scores, indices = self.index.search(q_emb, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            chunk = self.chunks[idx]
            results.append({**chunk, "score": float(score)})
        return results

vector_store = VectorStore()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def build_prompt(query, retrieved_chunks):
    context_blocks = []
    for i, c in enumerate(retrieved_chunks):
        tag = f"[{i+1}] source={c['source']} | type={c['doc_type']} | page={c['page_start']}"
        context_blocks.append(f"{tag}\n{c['text']}")
    context = "\n\n".join(context_blocks)

    prompt = f"""You are a helpful assistant answering questions using ONLY the provided context.

CONTEXT:
{context}

INSTRUCTIONS:
- Answer the question using only information found in the context above.
- If the context does not contain enough information to answer, say so clearly instead of guessing.
- Cite your sources inline using the bracketed numbers, e.g. [1], [2], matching the context blocks above.
- Be concise and factual.

QUESTION: {query}

ANSWER:"""
    return prompt

In [ ]:
def rag_answer(query, k=4):
    retrieved = vector_store.search(query, k=k)

    if not retrieved:
        return {
            "answer": "No documents have been indexed yet. Please upload a PDF first.",
            "sources": [],
            "confidence": 0.0,
            "chunk_count": 0,
        }

    prompt = build_prompt(query, retrieved)
    answer = call_llm(prompt)

    confidence = round(float(np.mean([c["score"] for c in retrieved])), 3)
    sources = [
        {
            "ref": f"[{i+1}]",
            "source": c["source"],
            "doc_type": c["doc_type"],
            "page": c["page_start"],
            "score": round(c["score"], 3),
        }
        for i, c in enumerate(retrieved)
    ]

    return {
        "answer": answer,
        "sources": sources,
        "confidence": confidence,
        "chunk_count": len(retrieved),
    }

In [1]:
import gradio as gr

uploaded_docs = []  # (path, source_id, doc_type)

def upload_and_index(files, doc_type):
    global uploaded_docs
    if not files:
        return "No files uploaded.", ""
    new_docs = [(f.name, os.path.basename(f.name), doc_type) for f in files]
    uploaded_docs.extend(new_docs)

    chunks = build_corpus(uploaded_docs)
    vector_store.build(chunks)
    return f"Indexed {len(uploaded_docs)} document(s), {len(chunks)} chunks total.", ""

def format_sources_markdown(sources):
    if not sources:
        return "_No sources retrieved._"
    lines = [f"**{s['ref']}** `{s['source']}` — {s['doc_type']}, page {s['page']} (relevance: {s['score']})" for s in sources]
    return "\n\n".join(lines)

def chat_fn(message, history):
    result = rag_answer(message)
    history = history + [(message, result["answer"])]
    sources_md = format_sources_markdown(result["sources"])
    meta_md = f"**Confidence:** {result['confidence']}  |  **Chunks used:** {result['chunk_count']}"
    return history, sources_md, meta_md

with gr.Blocks(title="Document Q&A Assistant") as demo:
    gr.Markdown("# 📄 Document Q&A Assistant\nUpload PDFs (digital or scanned) and ask questions grounded in their content.")

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="Upload PDF(s)", file_count="multiple", file_types=[".pdf"])
            doc_type_input = gr.Dropdown(
                choices=["Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration", "Cover Letter", "Other"],
                value="Other", label="Document Type"
            )
            index_btn = gr.Button("Process & Index", variant="primary")
            index_status = gr.Textbox(label="Indexing Status", interactive=False)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", height=400)
            msg_input = gr.Textbox(label="Ask a question", placeholder="e.g. What are the moisture test results?")
            send_btn = gr.Button("Send")
            sources_box = gr.Markdown(label="Sources")
            meta_box = gr.Markdown(label="Confidence & Chunk Count")

    index_btn.click(upload_and_index, inputs=[file_input, doc_type_input], outputs=[index_status, msg_input])
    send_btn.click(chat_fn, inputs=[msg_input, chatbot], outputs=[chatbot, sources_box, meta_box])
    msg_input.submit(chat_fn, inputs=[msg_input, chatbot], outputs=[chatbot, sources_box, meta_box])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e799fbd144a7baf022.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
